# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fkashaf19-afk/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one daily performance snapshot for a single content page: (client_id, content_id, report_date)
in fact_content_daily_performance. Table used: fact_content_daily_performance, month=2026-03 partition
(mid-panel, per the internship warning — the _sample table is the sealed final month and is never used
for label logic). Time window: 2026-03-01 to 2026-03-31, with features looking backward from each date
and the label looking 7 days forward from it. This mirrors the Lane 2 decision from w01: rank content
pages for a limited weekly review queue.

In [3]:
!pip -q install duckdb huggingface_hub

import os, duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    if "fact_content_daily_performance" in f and "sample" not in f:
        print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [4]:
FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT_PATH}')").show()
con.sql(f"SELECT * FROM read_parquet('{FACT_PATH}') LIMIT 5").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature — knowable before the decision moment:
  clicks_avg_7d_prior, gsc_position_avg_7d_prior, client_clicks_median_7d_prior,
  day_of_week, days_since_first_seen (all built only from data strictly before report_date)

Label / proxy — never a feature:
  is_declining_w03 = future 7-day avg clicks < prior 7-day avg clicks (the warehouse-native
  equivalent of w01's is_declining_label; there is no trend_direction column here, so this is
  built directly instead of inherited)

Context — for joining/grouping only, never learned from:
  content_id, client_id, report_date

Excluded — with reasons:
  future_clicks_7d itself — it IS the label's source, including it as a feature is the deliberate
  leak demonstrated in Section 3.
  ga4_* columns on rows where ga4_data_available IS FALSE — zero-filled, not real zeros.

In [5]:
# Confirm the fields actually exist before relying on them above
con.sql(f"""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM read_parquet('{FACT_PATH}'))
""").show()

┌────────────────────┬─────────────┐
│    column_name     │ column_type │
│      varchar       │   varchar   │
├────────────────────┼─────────────┤
│ report_date        │ DATE        │
│ client_hash_id     │ VARCHAR     │
│ content_hash_id    │ VARCHAR     │
│ client_has_gsc     │ BOOLEAN     │
│ client_has_ga4     │ BOOLEAN     │
│ gsc_data_available │ BOOLEAN     │
│ ga4_data_available │ BOOLEAN     │
│ gsc_impressions    │ BIGINT      │
│ gsc_clicks         │ BIGINT      │
│ gsc_sum_position   │ BIGINT      │
│      ·             │   ·         │
│      ·             │   ·         │
│      ·             │   ·         │
│ sessions_ai        │ BIGINT      │
│ ai_chatgpt         │ BIGINT      │
│ ai_perplexity      │ BIGINT      │
│ ai_gemini          │ BIGINT      │
│ ai_copilot         │ BIGINT      │
│ ai_claude          │ BIGINT      │
│ ai_meta            │ BIGINT      │
│ ai_other           │ BIGINT      │
│ scroll_events      │ BIGINT      │
│ month              │ VARCHAR     │
├

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries below (grain, count+span, availability), then the five-feature frame,
then the deliberate leak: adding future_clicks_7d directly as a feature, watching AUC jump toward 1.0,
then removing it and keeping the honest score.

In [6]:
# See ALL 31 columns (previous DESCRIBE truncated at 20)
con.sql(f"""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM read_parquet('{FACT_PATH}'))
""").df().to_string()

'                 column_name column_type\n0                report_date        DATE\n1             client_hash_id     VARCHAR\n2            content_hash_id     VARCHAR\n3             client_has_gsc     BOOLEAN\n4             client_has_ga4     BOOLEAN\n5         gsc_data_available     BOOLEAN\n6         ga4_data_available     BOOLEAN\n7            gsc_impressions      BIGINT\n8                 gsc_clicks      BIGINT\n9           gsc_sum_position      BIGINT\n10          gsc_avg_position      DOUBLE\n11             ga4_pageviews      BIGINT\n12              ga4_sessions      BIGINT\n13                 ga4_users      BIGINT\n14      ga4_engaged_sessions      BIGINT\n15  ga4_total_engagement_sec      BIGINT\n16          sessions_organic      BIGINT\n17           sessions_direct      BIGINT\n18         sessions_referral      BIGINT\n19           sessions_social      BIGINT\n20             sessions_paid      BIGINT\n21               sessions_ai      BIGINT\n22                ai_chatgpt     

In [7]:
# Query A — grain check (must return 0 rows)
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{FACT_PATH}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [8]:
# Query B — row count + date span
con.sql(f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{FACT_PATH}')
""").show()

┌─────────┬────────────┬────────────┐
│ n_rows  │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



In [9]:
# Query C — availability, IS TRUE
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc,
  ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
FROM read_parquet('{FACT_PATH}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬───────────────┐
│ total_rows │ rows_with_gsc │ pct_available │
│   int64    │    int128     │    double     │
├────────────┼───────────────┼───────────────┤
│    9841378 │       3611061 │          36.7 │
└────────────┴───────────────┴───────────────┘



In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

os.makedirs("work/outputs", exist_ok=True)
CACHE_PATH = "work/outputs/w03_march_slice.parquet"

if os.path.exists(CACHE_PATH):
    df = pd.read_parquet(CACHE_PATH)
else:
    df = con.sql(f"""
        SELECT * FROM read_parquet('{FACT_PATH}')
        WHERE gsc_data_available IS TRUE
    """).df()
    df.to_parquet(CACHE_PATH)

df["report_date"] = pd.to_datetime(df["report_date"])
df = df.sort_values(["content_hash_id", "report_date"])

# gsc_avg_position doesn't exist as a column — derive it
df["gsc_avg_position"] = df["gsc_sum_position"] / df["gsc_impressions"].replace(0, pd.NA)

# 1) clicks_avg_7d_prior — available when? shift(1) only looks at days BEFORE report_date
df["clicks_avg_7d_prior"] = df.groupby("content_hash_id")["gsc_clicks"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).mean())

# 2) gsc_position_avg_7d_prior — available when? same past-only window
df["gsc_position_avg_7d_prior"] = df.groupby("content_hash_id")["gsc_avg_position"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).mean())

# 3) day_of_week — available when? a calendar fact, known the moment report_date is fixed
df["day_of_week"] = df["report_date"].dt.dayofweek

# 4) client_clicks_median_7d_prior — available when? uses only the client's own PRIOR week
df["client_clicks_median_7d_prior"] = df.groupby("client_hash_id")["gsc_clicks"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).median())

# 5) days_since_first_seen — available when? computed from the earliest report_date already observed
df["first_seen"] = df.groupby("content_hash_id")["report_date"].transform("min")
df["days_since_first_seen"] = (df["report_date"] - df["first_seen"]).dt.days

# ---- label (future window, never a feature) ----
df["future_clicks_7d"] = df.groupby("content_hash_id")["gsc_clicks"].transform(
    lambda s: s.shift(-7).rolling(7, min_periods=1).mean())
df["is_declining_w03"] = (df["future_clicks_7d"] < df["clicks_avg_7d_prior"]).astype(int)

honest_feats = ["clicks_avg_7d_prior", "gsc_position_avg_7d_prior", "day_of_week",
                "client_clicks_median_7d_prior", "days_since_first_seen"]

# ---- deliberate leak ----
df["LEAK_future_clicks_7d"] = df["future_clicks_7d"]
leaky_feats = honest_feats + ["LEAK_future_clicks_7d"]

train_df = df.dropna(subset=leaky_feats + ["is_declining_w03"])

for name, feats in [("WITH leak", leaky_feats), ("honest (leak removed)", honest_feats)]:
    Xtr, Xte, ytr, yte = train_test_split(train_df[feats], train_df["is_declining_w03"],
                                           test_size=0.2, random_state=0)
    model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    print(f"{name}: AUC = {auc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice only covers content pages that have GSC data available in March 2026 — clients with
no search-console history in this month, or content pages that didn't exist yet, are systematically
excluded. It also can't tell us WHY clicks changed (algorithm update, seasonality, competitor
activity) — only that they did; "declining" here is a proxy label I chose, not ground truth
(same caveat as w01's careful-words section).

In [10]:
con.sql(f"""
SELECT
  SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
  SUM(CASE WHEN gsc_data_available IS FALSE THEN 1 ELSE 0 END) AS gsc_unavailable_rows
FROM read_parquet('{FACT_PATH}')
""").show()

┌────────────────────┬──────────────────────┐
│ gsc_available_rows │ gsc_unavailable_rows │
│       int128       │        int128        │
├────────────────────┼──────────────────────┤
│            3611061 │              6230317 │
└────────────────────┴──────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.